# 🧠 Word2Vec — Interactive Deep Dive

> **Goal:** By the end of this notebook, you will understand *exactly* what Word2Vec is, how it works internally, how to train it, and how to visualize the results — with intuitive analogies at every step.

---

## 📖 Table of Contents
1. [The Core Idea — What Problem Are We Solving?](#1)
2. [Setup & Imports](#2)
3. [Understanding the Context Window](#3)
4. [CBOW vs Skip-gram — The Two Architectures](#4)
5. [Training Word2Vec from Scratch (Toy Example)](#5)
6. [Training with Gensim on Real Text](#6)
7. [The Famous Vector Arithmetic (king − man + woman)](#7)
8. [Visualizing Word Embeddings with t-SNE](#8)
9. [Similarity Search](#9)
10. [Word2Vec Limitations](#10)

---

<a id='1'></a>
## 1. 🎯 The Core Idea — What Problem Are We Solving?

### The Old Way: One-Hot Encoding

Before Word2Vec, words were represented as **one-hot vectors**:

```
Vocabulary = [king, queen, man, woman, dog, cat]

king  = [1, 0, 0, 0, 0, 0]
queen = [0, 1, 0, 0, 0, 0]
man   = [0, 0, 1, 0, 0, 0]
dog   = [0, 0, 0, 0, 1, 0]
```

**Problems with this:**
- 🚫 Every word is equally distant from every other word
- 🚫 `king` and `queen` are as different as `king` and `dog`
- 🚫 Sparse, huge vectors (one dimension per word in vocabulary)
- 🚫 No notion of meaning or similarity

### The Word2Vec Way: Dense Embeddings

Word2Vec converts every word into a **small, dense vector** (e.g. 100 or 300 numbers) where:
- Similar words have **similar vectors**
- Relationships are encoded as **directions** in space
- The magic equation holds: `king − man + woman ≈ queen`

### 🚕 The Taxi Driver Analogy

> Imagine a taxi driver who never studied a city map, but drove **millions of trips**. After years of driving, they *know* the city — which streets connect, which neighborhoods are similar, which areas are near each other. They learned geography as a **side effect** of doing their job.
>
> Word2Vec is the taxi driver. Its "job" is predicting neighboring words. The city map it learns is the **semantic structure of language**.

### The Distributional Hypothesis

> *"You shall know a word by the company it keeps."* — J.R. Firth, 1957

Words that appear in **similar contexts** have **similar meanings**. Word2Vec operationalizes this idea.

<a id='2'></a>
## 2. 📦 Setup & Imports

In [ ]:
# ── Core libraries ──────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
import warnings
warnings.filterwarnings('ignore')

# ── NLP / ML ────────────────────────────────────────────────────────
from gensim.models import Word2Vec
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from collections import Counter, defaultdict
import re

# ── Plot style ──────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   '#f8f9fa',
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'font.size':        12,
})

print("✅ All libraries imported successfully!")
print(f"   NumPy  : {np.__version__}")
import gensim; print(f"   Gensim : {gensim.__version__}")

<a id='3'></a>
## 3. 🪟 Understanding the Context Window

The **context window** is the heart of Word2Vec. It defines what counts as a word's "neighborhood".

### 🔦 Spotlight Analogy

> Think of reading a book with a small spotlight that illuminates only a few words at a time. The center word is what you're studying; the surrounding words inside the spotlight are its context. Word2Vec slides this spotlight across the entire text.

For a window size of 2:
```
Sentence: "the quick brown fox jumps over the lazy dog"

Center: 'fox'
Context: ['quick', 'brown', 'jumps', 'over']  ← 2 left, 2 right
```

In [ ]:
def show_context_window(sentence, center_idx, window=2):
    """
    Visually display the context window around a center word.
    """
    words = sentence.split()
    n = len(words)
    
    print(f"\nSentence   : {sentence}")
    print(f"Center word: '{words[center_idx]}'  (window size = {window})")
    print()
    
    # Build the visual line
    display_words = []
    for i, w in enumerate(words):
        if i == center_idx:
            display_words.append(f"[**{w}**]")
        elif abs(i - center_idx) <= window:
            display_words.append(f"({w})")
        else:
            display_words.append(w)
    
    print("  " + "  ".join(display_words))
    print("\n  Legend: [**word**] = center,  (word) = context,  word = outside window")
    
    context = [words[i] for i in range(max(0, center_idx-window),
                                       min(n, center_idx+window+1))
               if i != center_idx]
    print(f"\n  Training pairs generated:")
    for c in context:
        print(f"    (center='{words[center_idx]}', context='{c}')")


sentence = "the quick brown fox jumps over the lazy dog"

print("=" * 60)
show_context_window(sentence, center_idx=3, window=2)   # fox
print("\n" + "=" * 60)
show_context_window(sentence, center_idx=3, window=1)   # fox, smaller window

In [ ]:
# ── Visualize ALL training pairs from a sentence ──────────────────
def generate_skipgram_pairs(sentence, window=2):
    words = sentence.lower().split()
    pairs = []
    for i, center in enumerate(words):
        for j in range(max(0, i-window), min(len(words), i+window+1)):
            if i != j:
                pairs.append((center, words[j]))
    return pairs

pairs = generate_skipgram_pairs("the quick brown fox jumps", window=2)

print("📋 All Skip-gram training pairs (center → context):")
print("-" * 40)
for p in pairs:
    print(f"  '{p[0]}' → '{p[1]}'")
print(f"\nTotal pairs: {len(pairs)}")

<a id='4'></a>
## 4. 🏗️ CBOW vs Skip-gram — The Two Architectures

Word2Vec comes in **two flavors**. Both learn word vectors, just from opposite directions.

---

### Architecture 1: CBOW (Continuous Bag of Words)

**Direction:** Context words → Predict center word

```
Input:  ['the', 'brown', 'jumps', 'over']  (context)
Output: 'fox'                               (center)
```

**📝 Fill-in-the-blank analogy:**
> "The quick ___ fox jumps over..."  
> Given the surrounding words, guess what's missing. That's CBOW.

---

### Architecture 2: Skip-gram

**Direction:** Center word → Predict context words

```
Input:  'fox'                               (center)
Output: ['the', 'brown', 'jumps', 'over']  (context)
```

**🏘️ Neighborhood prediction analogy:**
> If you know someone lives near a hospital, you'd predict neighbors include "doctors", "ambulances", "pharmacies". Given the word, predict its neighborhood.

---

| | CBOW | Skip-gram |
|---|---|---|
| **Input** | Context words | 1 center word |
| **Output** | 1 center word | Context words |
| **Speed** | ⚡ Faster | 🐢 Slower |
| **Best for** | Frequent words | Rare words |
| **Analogy** | Fill in the blank | Predict the neighborhood |

In [ ]:
# ── Visual comparison of CBOW vs Skip-gram ────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('CBOW vs Skip-gram Architecture', fontsize=16, fontweight='bold', y=1.02)

colors = {'input': '#4A90D9', 'hidden': '#7B68EE', 'output': '#F5A623'}

def draw_node(ax, x, y, text, color, radius=0.08, fontsize=10):
    circle = plt.Circle((x, y), radius, color=color, zorder=3)
    ax.add_patch(circle)
    ax.text(x, y, text, ha='center', va='center', fontsize=fontsize,
            color='white', fontweight='bold', zorder=4)

def draw_arrow(ax, x1, y1, x2, y2, color='#666'):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color=color, lw=1.5))

# ── CBOW ──
ax = axes[0]
ax.set_xlim(0, 1); ax.set_ylim(-0.1, 1.1)
ax.set_aspect('equal'); ax.axis('off')
ax.set_title('CBOW\n(Context → Center Word)', fontsize=13, pad=15)

context_y = [0.2, 0.4, 0.6, 0.8]
labels = ['over', 'jumps', 'brown', 'the']
for y, lbl in zip(context_y, labels):
    draw_node(ax, 0.2, y, lbl, colors['input'])
    draw_arrow(ax, 0.29, y, 0.46, 0.5)

draw_node(ax, 0.5, 0.5, 'avg\nvec', colors['hidden'], radius=0.1, fontsize=9)
draw_arrow(ax, 0.61, 0.5, 0.75, 0.5)
draw_node(ax, 0.8, 0.5, 'fox', colors['output'])

ax.text(0.2, 0.05, 'Context\nwords', ha='center', fontsize=10, color=colors['input'])
ax.text(0.5, 0.35, 'Hidden\nlayer', ha='center', fontsize=10, color=colors['hidden'])
ax.text(0.8, 0.35, 'Target\nword', ha='center', fontsize=10, color=colors['output'])

# ── Skip-gram ──
ax2 = axes[1]
ax2.set_xlim(0, 1); ax2.set_ylim(-0.1, 1.1)
ax2.set_aspect('equal'); ax2.axis('off')
ax2.set_title('Skip-gram\n(Center Word → Context)', fontsize=13, pad=15)

draw_node(ax2, 0.2, 0.5, 'fox', colors['output'])
draw_arrow(ax2, 0.29, 0.5, 0.4, 0.5)
draw_node(ax2, 0.5, 0.5, 'vec', colors['hidden'], radius=0.1, fontsize=9)

for y, lbl in zip(context_y, labels):
    draw_arrow(ax2, 0.61, 0.5, 0.72, y)
    draw_node(ax2, 0.8, y, lbl, colors['input'])

ax2.text(0.2, 0.35, 'Input\nword', ha='center', fontsize=10, color=colors['output'])
ax2.text(0.5, 0.35, 'Hidden\nlayer', ha='center', fontsize=10, color=colors['hidden'])
ax2.text(0.8, 0.05, 'Predicted\ncontext', ha='center', fontsize=10, color=colors['input'])

plt.tight_layout()
plt.savefig('cbow_vs_skipgram.png', dpi=120, bbox_inches='tight')
plt.show()
print("\n💡 Key insight: Both architectures learn the SAME weight matrix — the word vectors.")
print("   The 'prediction task' is just a training vehicle. The vectors are the real prize.")

<a id='5'></a>
## 5. 🔩 Training Word2Vec from Scratch (Toy Example)

Let's implement the **core forward pass** of Skip-gram manually to see what's happening inside the neural network.

### 🏗️ The Neural Network Structure

```
Input layer  : one-hot vector  [V × 1]
     ↓  (multiply by W_embed)
Hidden layer : word embedding  [N × 1]   ← THIS is the word vector we want!
     ↓  (multiply by W_context)
Output layer : softmax scores  [V × 1]
```

Where V = vocabulary size, N = embedding dimensions

In [ ]:
# ── Step 1: Build vocabulary from toy corpus ──────────────────────
corpus = [
    "the king ruled the kingdom with wisdom",
    "the queen ruled the kingdom with grace",
    "the king and queen wore golden crowns",
    "the man wore a crown on his head",
    "the woman wore a crown on her head",
    "the dog barked at the cat in the garden",
    "the cat slept near the garden fence",
    "the prince will become king one day",
    "the princess will become queen one day",
]

# Tokenize
tokens = [word for sent in corpus for word in sent.lower().split()]
vocab  = sorted(set(tokens))
V      = len(vocab)

word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}

print(f"Corpus stats:")
print(f"  Sentences : {len(corpus)}")
print(f"  Tokens    : {len(tokens)}")
print(f"  Vocabulary: {V} unique words")
print(f"\nVocabulary:")
print("  " + ", ".join(vocab))

In [ ]:
# ── Step 2: One-hot encoding ───────────────────────────────────────
def one_hot(word, vocab_size, word2idx):
    """Create a one-hot vector for a word."""
    vec = np.zeros(vocab_size)
    vec[word2idx[word]] = 1
    return vec

# Demo
demo_word = 'king'
oh = one_hot(demo_word, V, word2idx)
print(f"One-hot vector for '{demo_word}':")
print(f"  Length: {len(oh)}  (one slot per vocabulary word)")
print(f"  Non-zero index: {np.where(oh==1)[0][0]}")
print(f"  Vector (first 10 dims): {oh[:10]}")
print(f"\n💡 Notice: This is a 'sparse' representation — mostly zeros.")
print(f"   Word2Vec will compress this into just {min(V, 10)} meaningful numbers.")

In [ ]:
# ── Step 3: Initialize weight matrices ───────────────────────────
np.random.seed(42)
N = 10  # embedding dimensions (small for toy example)

# W_embed  : shape [V, N] — the embedding matrix (rows = word vectors)
# W_context: shape [N, V] — the output weight matrix
W_embed   = np.random.randn(V, N) * 0.01
W_context = np.random.randn(N, V) * 0.01

print("Initialized weight matrices:")
print(f"  W_embed   shape: {W_embed.shape}   (vocabulary × embedding_dims)")
print(f"  W_context shape: {W_context.shape}  (embedding_dims × vocabulary)")
print(f"\n  Total parameters: {W_embed.size + W_context.size:,}")
print(f"\n💡 The ROWS of W_embed are the word vectors we want!")
print(f"   W_embed['king'] = W_embed[{word2idx['king']}] = {W_embed[word2idx['king']].round(4)}")

In [ ]:
# ── Step 4: Forward pass + Softmax ───────────────────────────────
def softmax(x):
    e = np.exp(x - x.max())
    return e / e.sum()

def forward_pass(center_word, W_embed, W_context, word2idx):
    """
    Skip-gram forward pass.
    Returns: hidden vector, output probabilities
    """
    # 1. Look up the embedding (same as: one_hot(word) @ W_embed)
    h = W_embed[word2idx[center_word]]          # shape: [N]
    
    # 2. Compute scores for all vocabulary words
    scores = W_context.T @ h                   # shape: [V]
    
    # 3. Convert to probabilities
    probs = softmax(scores)                     # shape: [V]
    
    return h, probs

# Demo
h, probs = forward_pass('king', W_embed, W_context, word2idx)

print("Forward pass for center word = 'king':")
print(f"\n1. Hidden vector (embedding of 'king'):")
print(f"   {h.round(5)}")
print(f"\n2. Top 5 predicted context words (before training — random!):")
top5 = np.argsort(probs)[::-1][:5]
for i in top5:
    print(f"   '{idx2word[i]}': {probs[i]:.4f}")
print("\n💡 These predictions are random right now — training will fix that.")

In [ ]:
# ── Step 5: Training loop ─────────────────────────────────────────
def train_skipgram(corpus, word2idx, idx2word, V, N=10, window=2,
                   lr=0.05, epochs=500):
    """
    Train Skip-gram Word2Vec from scratch.
    Returns: embedding matrix, loss history
    """
    np.random.seed(42)
    W_emb = np.random.randn(V, N) * 0.01
    W_ctx = np.random.randn(N, V) * 0.01
    losses = []

    for epoch in range(epochs):
        total_loss = 0

        for sentence in corpus:
            words = sentence.lower().split()
            for i, center in enumerate(words):
                if center not in word2idx:
                    continue
                ci = word2idx[center]

                for j in range(max(0, i-window), min(len(words), i+window+1)):
                    if i == j or words[j] not in word2idx:
                        continue
                    ctx_i = word2idx[words[j]]

                    # Forward pass
                    h     = W_emb[ci]                  # [N]
                    score = W_ctx.T @ h                # [V]
                    prob  = softmax(score)             # [V]

                    # Loss (cross-entropy)
                    total_loss -= np.log(prob[ctx_i] + 1e-9)

                    # Backward pass
                    dout          = prob.copy()
                    dout[ctx_i]  -= 1                  # gradient of softmax + cross-entropy

                    dW_ctx = np.outer(h, dout)         # [N, V]
                    dh     = W_ctx @ dout              # [N]

                    # Update
                    W_ctx -= lr * dW_ctx
                    W_emb[ci] -= lr * dh

        losses.append(total_loss)
        if (epoch + 1) % 100 == 0:
            print(f"  Epoch {epoch+1:4d}/{epochs} | Loss: {total_loss:.2f}")

    return W_emb, W_ctx, losses

print("🚀 Training Skip-gram from scratch...")
W_trained, W_ctx_trained, losses = train_skipgram(
    corpus, word2idx, idx2word, V, N=10, epochs=500, lr=0.05
)
print("\n✅ Training complete!")

In [ ]:
# ── Plot training loss ────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(losses, color='#4A90D9', linewidth=2)
ax.fill_between(range(len(losses)), losses, alpha=0.15, color='#4A90D9')
ax.set_xlabel('Epoch')
ax.set_ylabel('Cross-Entropy Loss')
ax.set_title('Training Loss (Skip-gram from Scratch)', fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print("💡 Loss decreasing = model is getting better at predicting context words.")

In [ ]:
# ── Check similarity from scratch model ──────────────────────────
from numpy.linalg import norm

def cosine_similarity(v1, v2):
    return np.dot(v1, v2) / (norm(v1) * norm(v2) + 1e-9)

def most_similar_scratch(word, W, word2idx, idx2word, top_n=5):
    if word not in word2idx:
        print(f"'{word}' not in vocabulary"); return
    vec = W[word2idx[word]]
    sims = [(idx2word[i], cosine_similarity(vec, W[i]))
            for i in range(len(idx2word)) if idx2word[i] != word]
    sims.sort(key=lambda x: -x[1])
    return sims[:top_n]

print("📊 Similarity results from our scratch model (small corpus, so imperfect):")
for query in ['king', 'dog', 'queen']:
    sims = most_similar_scratch(query, W_trained, word2idx, idx2word)
    print(f"\n  Most similar to '{query}':")
    for w, s in sims:
        bar = '█' * int(s * 20) if s > 0 else ''
        print(f"    {w:12s} {s:+.4f}  {bar}")

<a id='6'></a>
## 6. 🏭 Training with Gensim on Real Text

Our scratch implementation shows the mechanics. In practice, we use **Gensim**, which includes:
- **Negative Sampling** (much faster than full softmax)
- **Subsampling** of frequent words
- **Hierarchical Softmax** option
- Multi-core training

In [ ]:
# ── Build a richer corpus ─────────────────────────────────────────
rich_corpus_raw = """
the king ruled the kingdom with great wisdom and power
the queen ruled beside the king with grace and intelligence
the prince is the son of the king and will become king
the princess is the daughter of the queen
a man worked hard every day to feed his family
a woman worked hard to raise her children with love
the king and queen lived in a magnificent royal palace
the royal family attended grand celebrations in the palace
a dog barked loudly at the cat sleeping near the fence
the cat chased the small mouse across the garden
the dog and the cat are common household pets
wolf and fox are wild animals that live in forests
the doctor treated the patient in the hospital
the nurse helped the doctor care for the patient
the teacher taught students in the school classroom
the student studied mathematics and science at school
paris is the capital city of france in europe
berlin is the capital city of germany in europe
london is the capital city of england in europe
madrid is the capital city of spain in europe
france germany england and spain are european countries
the scientist discovered a new formula in the laboratory
the engineer built a bridge over the wide river
music fills the concert hall with beautiful sound
the artist painted a beautiful picture in bright colors
the writer wrote a long novel about love and war
the chef cooked delicious food in the restaurant kitchen
the sun rises in the east and sets in the west
the moon shines brightly in the dark night sky
rivers flow from mountains down into the ocean sea
trees and flowers grow in the green forest and garden
the athlete ran fast and won the championship race
football basketball and tennis are popular sports worldwide
computers and smartphones are modern technology devices
the internet connects millions of people around the world
gold and silver are precious metals found in the earth
bread and butter are common foods eaten at breakfast
coffee and tea are popular hot drinks in the morning
the happy child played in the warm sunny garden
the sad man sat alone on the cold empty bench
"""

# Tokenize into list of sentences
sentences = [re.findall(r'\w+', line.lower())
             for line in rich_corpus_raw.strip().split('\n') if line.strip()]

all_words = [w for s in sentences for w in s]
print(f"Rich corpus stats:")
print(f"  Sentences   : {len(sentences)}")
print(f"  Total tokens: {len(all_words)}")
print(f"  Unique words: {len(set(all_words))}")

In [ ]:
# ── Train Word2Vec with Gensim ─────────────────────────────────────
# sg=0: CBOW,  sg=1: Skip-gram
# window : context window size
# vector_size: embedding dimensions
# min_count   : ignore words with fewer occurrences
# workers     : number of CPU cores

model_sg = Word2Vec(
    sentences,
    sg=1,           # Skip-gram
    vector_size=50,
    window=3,
    min_count=1,
    workers=2,
    epochs=200,
    seed=42
)

model_cbow = Word2Vec(
    sentences,
    sg=0,           # CBOW
    vector_size=50,
    window=3,
    min_count=1,
    workers=2,
    epochs=200,
    seed=42
)

print("✅ Both models trained!")
print(f"\n  Vocabulary size: {len(model_sg.wv.key_to_index)}")
print(f"  Embedding dims : {model_sg.wv.vector_size}")
print(f"\n  Vector for 'king' (first 8 dims):")
print(f"  {model_sg.wv['king'][:8].round(4)}")

In [ ]:
# ── Gensim key parameters explained ──────────────────────────────
params = {
    'vector_size': ('Embedding dimensions', '50–300 typical; bigger = richer but slower'),
    'window'     : ('Context window size',  '2–5; bigger = broader topic context'),
    'sg'         : ('Algorithm',            '0=CBOW, 1=Skip-gram'),
    'min_count'  : ('Min word frequency',   'Ignore rare words (reduces noise)'),
    'negative'   : ('Negative samples',     '5–20; how many "wrong" samples per positive'),
    'alpha'      : ('Learning rate',        '0.025 default; decreases during training'),
    'epochs'     : ('Training epochs',      'More = better on small corpus'),
}

print("📋 Gensim Word2Vec Key Parameters:")
print("-" * 65)
print(f"  {'Parameter':<15} {'What it does':<25} {'Guidance'}")
print("-" * 65)
for param, (what, guidance) in params.items():
    print(f"  {param:<15} {what:<25} {guidance}")

<a id='7'></a>
## 7. ➕➖ The Famous Vector Arithmetic

> **king − man + woman ≈ queen**

This isn't magic — it's geometry! Word vectors encode **relational directions** in space.

The vector from `man → king` encodes "royalty". The same vector applied to `woman` gives `queen`.

### 🗺️ Map Analogy
> Paris is to France as Berlin is to Germany.  
> `Paris − France + Germany ≈ Berlin`  
> The direction "capital of" is consistent across all country pairs.

In [ ]:
# ── Vector Arithmetic ─────────────────────────────────────────────
wv = model_sg.wv

def vector_analogy(positive_words, negative_words, model_wv, top_n=5):
    """
    Perform: result ≈ sum(positive) − sum(negative)
    """
    try:
        results = model_wv.most_similar(
            positive=positive_words,
            negative=negative_words,
            topn=top_n
        )
        return results
    except KeyError as e:
        print(f"Word not in vocabulary: {e}"); return []

print("🔮 Vector Arithmetic Analogies:")
print("=" * 55)

analogies = [
    (['king', 'woman'], ['man'],   "king − man + woman ≈ ?"),
    (['queen', 'man'],  ['woman'], "queen − woman + man ≈ ?"),
    (['doctor', 'woman'], ['man'], "doctor − man + woman ≈ ?"),
    (['paris', 'germany'], ['france'], "paris − france + germany ≈ ?"),
]

for pos, neg, desc in analogies:
    print(f"\n  {desc}")
    results = vector_analogy(pos, neg, wv, top_n=3)
    if results:
        for word, score in results:
            bar = '█' * int(score * 30)
            print(f"    → '{word}' (similarity: {score:.4f}) {bar}")
    else:
        print("    (not enough training data for this analogy)")

In [ ]:
# ── Visualize the vector offset ───────────────────────────────────
words_to_plot = ['king', 'queen', 'man', 'woman', 'prince', 'princess']
available = [w for w in words_to_plot if w in wv.key_to_index]

vectors = np.array([wv[w] for w in available])

# Reduce to 2D with PCA
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(vectors)

fig, ax = plt.subplots(figsize=(9, 7))
ax.set_facecolor('#f8f9fa')

colors_map = {
    'king': '#4A90D9', 'prince': '#4A90D9',
    'queen': '#E87070', 'princess': '#E87070',
    'man': '#7B68EE', 'woman': '#F5A623'
}

for i, word in enumerate(available):
    x, y = coords[i]
    color = colors_map.get(word, '#888')
    ax.scatter(x, y, s=200, color=color, zorder=3, alpha=0.9)
    ax.annotate(word, (x, y), fontsize=13, fontweight='bold',
                xytext=(8, 8), textcoords='offset points', color=color)

# Draw gender offset arrow (man → woman, king → queen)
if 'king' in available and 'queen' in available:
    ki = available.index('king'); qi = available.index('queen')
    ax.annotate('', xy=coords[qi], xytext=coords[ki],
                arrowprops=dict(arrowstyle='->', color='#999', lw=1.5, linestyle='dashed'))

if 'man' in available and 'woman' in available:
    mi = available.index('man'); wi = available.index('woman')
    ax.annotate('', xy=coords[wi], xytext=coords[mi],
                arrowprops=dict(arrowstyle='->', color='#999', lw=1.5, linestyle='dashed'))

ax.set_title('Word Vectors in 2D (PCA) — Royalty & Gender', fontweight='bold', fontsize=14)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')

legend_elements = [
    mpatches.Patch(color='#4A90D9', label='Male royalty'),
    mpatches.Patch(color='#E87070', label='Female royalty'),
    mpatches.Patch(color='#7B68EE', label='man'),
    mpatches.Patch(color='#F5A623', label='woman'),
]
ax.legend(handles=legend_elements, loc='best', fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

<a id='8'></a>
## 8. 🗺️ Visualizing Word Embeddings with t-SNE

**t-SNE** (t-Distributed Stochastic Neighbor Embedding) is the standard tool for visualizing high-dimensional embeddings in 2D. It preserves *local structure* — nearby words in the embedding space stay nearby in the 2D plot.

### 📍 City Map Analogy
> t-SNE is like flattening a globe onto a flat map. You lose some long-range accuracy (like map projections distort area), but nearby cities stay near each other. That local structure is what matters.

In [ ]:
# ── t-SNE visualization ───────────────────────────────────────────
word_groups = {
    'Royalty':    ['king', 'queen', 'prince', 'princess', 'royal', 'crown'],
    'People':     ['man', 'woman', 'child', 'family'],
    'Animals':    ['dog', 'cat', 'wolf', 'fox', 'mouse'],
    'Professions':['doctor', 'nurse', 'teacher', 'scientist', 'engineer', 'chef'],
    'Countries':  ['france', 'germany', 'england', 'spain'],
    'Cities':     ['paris', 'berlin', 'london', 'madrid'],
    'Nature':     ['sun', 'moon', 'river', 'forest', 'garden', 'tree'],
}

group_colors = {
    'Royalty':     '#F5A623',
    'People':      '#7B68EE',
    'Animals':     '#4CAF50',
    'Professions': '#4A90D9',
    'Countries':   '#E87070',
    'Cities':      '#E87070',
    'Nature':      '#26A69A',
}

# Collect available words
plot_words  = []
plot_groups = []
for group, words in word_groups.items():
    for w in words:
        if w in wv.key_to_index:
            plot_words.append(w)
            plot_groups.append(group)

plot_vecs = np.array([wv[w] for w in plot_words])

# t-SNE
tsne = TSNE(n_components=2, random_state=42, perplexity=min(15, len(plot_words)-1),
            n_iter=2000, learning_rate='auto', init='random')
coords_2d = tsne.fit_transform(plot_vecs)

# Plot
fig, ax = plt.subplots(figsize=(13, 9))
ax.set_facecolor('#f8f9fa')

for i, (word, group) in enumerate(zip(plot_words, plot_groups)):
    x, y = coords_2d[i]
    color = group_colors[group]
    ax.scatter(x, y, s=120, color=color, alpha=0.85, zorder=3)
    ax.annotate(word, (x, y), fontsize=10,
                xytext=(5, 5), textcoords='offset points', color=color, fontweight='bold')

# Legend
unique_groups = list(dict.fromkeys(plot_groups))
patches = [mpatches.Patch(color=group_colors[g], label=g) for g in unique_groups]
ax.legend(handles=patches, loc='best', fontsize=11, title='Word Groups')

ax.set_title('Word2Vec Embeddings Visualized with t-SNE\n(Similar words cluster together)', 
             fontweight='bold', fontsize=14)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('word2vec_tsne.png', dpi=120, bbox_inches='tight')
plt.show()

print("💡 Notice how words cluster by meaning — even though the model never")
print("   learned what 'royalty' or 'animals' means. It inferred structure from context.")

<a id='9'></a>
## 9. 🔍 Similarity Search

Word2Vec measures similarity using **cosine similarity** — the angle between two vectors.

```
cos(θ) = (A · B) / (|A| × |B|)

  1.0  = identical direction (same word / synonym)
  0.0  = orthogonal (unrelated)
 -1.0  = opposite direction (antonyms)
```

### 🧭 Compass Analogy
> Think of each word as a compass pointing in a direction. Cosine similarity measures how closely two compasses align — not how far apart they are, but whether they're pointing the same way.

In [ ]:
# ── Cosine Similarity Explained ───────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

cases = [
    ('king', 'queen', '#F5A623', 'High similarity\n(related concepts)'),
    ('king', 'doctor', '#4A90D9', 'Low similarity\n(unrelated)'),
    ('man', 'woman', '#7B68EE', 'Medium similarity\n(same category, different)'),
]

for ax, (w1, w2, color, title) in zip(axes, cases):
    if w1 in wv.key_to_index and w2 in wv.key_to_index:
        v1 = wv[w1]; v2 = wv[w2]
        sim = np.dot(v1, v2) / (norm(v1) * norm(v2))

        # Project to 2D for illustration
        pca2 = PCA(n_components=2).fit_transform(np.array([v1, v2]))
        p1, p2 = pca2[0], pca2[1]

        ax.set_xlim(-2, 2); ax.set_ylim(-2, 2)
        ax.set_aspect('equal'); ax.set_facecolor('#f8f9fa')
        ax.axhline(0, color='#ccc', lw=0.8); ax.axvline(0, color='#ccc', lw=0.8)

        ax.annotate('', xy=p1/max(norm(p1),1e-9)*1.5, xytext=(0, 0),
                    arrowprops=dict(arrowstyle='->', color=color, lw=2.5))
        ax.annotate('', xy=p2/max(norm(p2),1e-9)*1.5, xytext=(0, 0),
                    arrowprops=dict(arrowstyle='->', color='#555', lw=2.5))

        ax.text(p1[0]/max(norm(p1),1e-9)*1.7, p1[1]/max(norm(p1),1e-9)*1.7, f"'{w1}'",
                ha='center', fontsize=11, color=color, fontweight='bold')
        ax.text(p2[0]/max(norm(p2),1e-9)*1.7, p2[1]/max(norm(p2),1e-9)*1.7, f"'{w2}'",
                ha='center', fontsize=11, color='#555', fontweight='bold')

        ax.set_title(f"{title}\ncosine = {sim:.3f}", fontsize=10, fontweight='bold')
        ax.grid(True, alpha=0.2)

plt.suptitle('Cosine Similarity Between Word Vectors', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Interactive similarity explorer ──────────────────────────────
def explore_word(word, model_wv, topn=8):
    if word not in model_wv.key_to_index:
        print(f"'{word}' not in vocabulary. Available: {list(model_wv.key_to_index.keys())[:20]}")
        return

    similar = model_wv.most_similar(word, topn=topn)

    print(f"\n🔍 Top {topn} words most similar to '{word}':")
    print("-" * 45)

    fig, ax = plt.subplots(figsize=(9, 4))
    words_s = [w for w, _ in similar]
    scores  = [s for _, s in similar]

    bars = ax.barh(words_s[::-1], scores[::-1], color='#4A90D9', alpha=0.8)
    ax.set_xlim(0, 1)
    ax.set_xlabel('Cosine Similarity', fontsize=12)
    ax.set_title(f"Words most similar to '{word}'", fontweight='bold', fontsize=13)
    ax.axvline(0.5, color='#F5A623', linestyle='--', alpha=0.6, label='0.5 threshold')

    for bar, score in zip(bars, scores[::-1]):
        ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                f'{score:.3f}', va='center', fontsize=10)

    ax.grid(True, alpha=0.3, axis='x')
    ax.legend()
    plt.tight_layout()
    plt.show()

    for w, s in similar:
        bar_str = '█' * int(s * 25)
        print(f"  {w:<15} {s:.4f}  {bar_str}")

# 🎮 Try different words here!
explore_word('king', wv)
explore_word('doctor', wv)

In [ ]:
# ── Pairwise similarity heatmap ───────────────────────────────────
interest_words = ['king', 'queen', 'man', 'woman', 'dog', 'cat',
                  'doctor', 'nurse', 'paris', 'france']
available_iw   = [w for w in interest_words if w in wv.key_to_index]

sim_matrix = np.zeros((len(available_iw), len(available_iw)))
for i, w1 in enumerate(available_iw):
    for j, w2 in enumerate(available_iw):
        sim_matrix[i, j] = wv.similarity(w1, w2)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(sim_matrix, cmap='RdYlGn', vmin=-0.2, vmax=1.0, aspect='auto')
plt.colorbar(im, ax=ax, label='Cosine Similarity')

ax.set_xticks(range(len(available_iw)))
ax.set_yticks(range(len(available_iw)))
ax.set_xticklabels(available_iw, rotation=45, ha='right', fontsize=11)
ax.set_yticklabels(available_iw, fontsize=11)

for i in range(len(available_iw)):
    for j in range(len(available_iw)):
        val = sim_matrix[i, j]
        color = 'white' if abs(val) > 0.6 else 'black'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=9, color=color)

ax.set_title('Pairwise Cosine Similarity Heatmap\n(Green = similar, Red = dissimilar)',
             fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('similarity_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()

<a id='10'></a>
## 10. ⚠️ Word2Vec Limitations

Understanding limitations is just as important as understanding strengths.

| Limitation | Example | Fixed by |
|---|---|---|
| **One vector per word** | `bank` (river) = `bank` (money) | BERT, ELMo |
| **No sub-word info** | `run`, `running`, `runner` unrelated | FastText |
| **Context-free** | Meaning fixed regardless of sentence | BERT |
| **Needs lots of data** | Rare words get poor vectors | BPE tokenization |
| **Static** | Can't update for new words | Fine-tuning LLMs |

In [ ]:
# ── Demonstrate the polysemy problem ─────────────────────────────
print("⚠️  LIMITATION: Word2Vec treats polysemous words as a single point.")
print()
print("  'bank' in 'river bank'      → financial? nature?")
print("  'bank' in 'bank account'    → Same vector! Word2Vec can't distinguish.")
print()
print("  Solution: BERT computes a DIFFERENT vector for 'bank'")
print("  depending on the sentence context (contextual embeddings).")
print()
print("─" * 55)
print()
print("📊 Summary: When to use Word2Vec vs alternatives")
print()
models_comparison = [
    ('Word2Vec',  'Fast, simple, great for search/similarity', 'No context, polysemy issue'),
    ('FastText',  'Handles rare words, morphology',            'Still context-free'),
    ('GloVe',     'Global co-occurrence, good analogies',      'Same context issues'),
    ('BERT/GPT',  'Contextual, state-of-the-art',              'Slow, expensive'),
]
print(f"  {'Model':<12} {'Strength':<42} {'Weakness'}")
print("  " + "-" * 80)
for model, strength, weakness in models_comparison:
    print(f"  {model:<12} {strength:<42} {weakness}")

print()
print("✅ Word2Vec is still widely used as a FAST baseline and for")
print("   recommendation systems, search engines, and NLP pipelines.")

---

## 🎓 Summary

| Concept | Key Takeaway |
|---|---|
| **Core idea** | Words with similar contexts have similar vectors |
| **CBOW** | Predict center word from context (fill-in-the-blank) |
| **Skip-gram** | Predict context from center word (predict neighborhood) |
| **Training** | Fake prediction task → real embeddings as side effect |
| **Vector arithmetic** | king − man + woman ≈ queen |
| **Similarity** | Cosine similarity measures alignment of vectors |
| **Visualization** | t-SNE reveals semantic clusters in 2D |
| **Limitation** | One vector per word (no context sensitivity) |

---

### 🔗 What's Next?
- **GloVe** — global co-occurrence matrix approach (see the companion notebook!)
- **FastText** — adds subword (character n-gram) embeddings
- **BERT** — contextual embeddings that solve polysemy
- **Sentence Transformers** — embeddings for whole sentences